<a href="https://colab.research.google.com/github/developer123-crypto/ai-study-planner/blob/main/AI_StudyPlanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 AI Study Planner
Free schedule generator + AI study tips (Gemini / Gemma).

**Setup:** add `GEMINI_API_KEY` in Colab **Secrets** (🔑 icon, left side) and turn on *Notebook access*. The key is optional: the schedule works without it.

In [22]:
import os
try:
    from google.colab import userdata
    try:
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        print("GEMINI_API_KEY not set (AI tips will be skipped)")
except ImportError:
    pass  # running outside Colab: use an environment variable


## 1. Planner code

In [23]:
"""Core scheduling logic. Pure Python: no API, no cost."""
import csv
from dataclasses import dataclass
from datetime import date, timedelta

DIFFICULTY = {"easy": 1, "medium": 2, "hard": 3}


@dataclass
class Subject:
    name: str
    exam_date: date
    difficulty: int = 2  # 1 = easy, 2 = medium, 3 = hard


def _split_hours(total, weights, step=0.5):
    """Split `total` hours by weight in `step` blocks (largest remainder)."""
    units = int(round(total / step))
    wsum = sum(weights.values())
    if units == 0 or wsum == 0:
        return {}
    raw = {k: units * w / wsum for k, w in weights.items()}
    base = {k: int(v) for k, v in raw.items()}
    left = units - sum(base.values())
    for k in sorted(raw, key=lambda k: raw[k] - base[k], reverse=True)[:left]:
        base[k] += 1
    return {k: v * step for k, v in base.items() if v > 0}


def generate_schedule(subjects, hours_per_day, start=None):
    """Return a list of {date, subject, hours} rows.

    Weight for each subject on each day = difficulty / (days_left + 1),
    so harder subjects and closer exams get more time.
    """
    start = start or date.today()
    if not subjects:
        raise ValueError("Add at least one subject.")
    if hours_per_day <= 0:
        raise ValueError("Hours per day must be greater than 0.")
    for s in subjects:
        if s.difficulty not in (1, 2, 3):
            raise ValueError(f"{s.name}: difficulty must be 1, 2 or 3.")
        if s.exam_date < start:
            raise ValueError(f"{s.name}: exam date is in the past.")

    rows, day = [], start
    last = max(s.exam_date for s in subjects)
    while day <= last:
        active = [s for s in subjects if s.exam_date >= day]
        weights = {s.name: s.difficulty / ((s.exam_date - day).days + 1) for s in active}
        for name, hrs in _split_hours(hours_per_day, weights).items():
            rows.append({"date": day, "subject": name, "hours": hrs})
        day += timedelta(days=1)
    return rows


def summarize(rows):
    """Total hours per subject."""
    out = {}
    for r in rows:
        out[r["subject"]] = out.get(r["subject"], 0) + r["hours"]
    return out


def export_csv(rows, path="study_plan.csv"):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["date", "subject", "hours"])
        w.writeheader()
        w.writerows(rows)
    return path


def export_ics(rows, path="study_plan.ics"):
    """All-day calendar events you can import into Google Calendar."""
    lines = ["BEGIN:VCALENDAR", "VERSION:2.0", "PRODID:-//ai-study-planner//EN"]
    for i, r in enumerate(rows):
        d = r["date"]
        nxt = d + timedelta(days=1)
        lines += [
            "BEGIN:VEVENT",
            f"UID:study-{i}-{d:%Y%m%d}@ai-study-planner",
            f"DTSTAMP:{d:%Y%m%d}T000000Z",
            f"DTSTART;VALUE=DATE:{d:%Y%m%d}",
            f"DTEND;VALUE=DATE:{nxt:%Y%m%d}",
            f"SUMMARY:Study {r['subject']} ({r['hours']}h)",
            "END:VEVENT",
        ]
    lines.append("END:VCALENDAR")
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write("\r\n".join(lines))
    return path


## 2. AI code (Gemini free tier)

In [24]:
"""Free AI tips using Gemini (Google AI Studio free key).

Tries a list of Gemini models; if all fail, asks Google which models your key can
use and tries those. Shows the exact error for every model that failed."""
import os
import requests

BASE = "https://generativelanguage.googleapis.com/v1beta"
PREFERRED = ["gemini-flash-latest", "gemini-2.5-flash", "gemini-2.0-flash", "gemini-flash-lite-latest"]
SKIP = ("tts", "image", "live", "audio", "embedding", "robotics", "computer-use", "veo", "imagen")


def _headers(key):
    return {"x-goog-api-key": key, "Content-Type": "application/json"}


def available_models(key, timeout=30):
    """Models your key can use for text generation."""
    r = requests.get(f"{BASE}/models", headers=_headers(key), params={"pageSize": 200}, timeout=timeout)
    r.raise_for_status()
    names = []
    for m in r.json().get("models", []):
        if "generateContent" in m.get("supportedGenerationMethods", []):
            names.append(m["name"].split("/", 1)[1])
    return names


def _call(model, prompt, key, timeout):
    r = requests.post(
        f"{BASE}/models/{model}:generateContent",
        headers=_headers(key),
        json={"contents": [{"parts": [{"text": prompt}]}]},
        timeout=timeout,
    )
    if r.status_code != 200:
        raise RuntimeError(f"{model}: HTTP {r.status_code} {r.text[:200]}")
    return r.json()["candidates"][0]["content"]["parts"][0]["text"].strip()


def ask_gemini(prompt, models=None, timeout=60):
    """Returns (model_name, text)."""
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        raise RuntimeError("GEMINI_API_KEY not set")
    errors, tried = [], []

    def attempt(model):
        tried.append(model)
        try:
            return model, _call(model, prompt, key, timeout)
        except Exception as e:
            errors.append(str(e))
            return None

    for m in models or PREFERRED:
        res = attempt(m)
        if res:
            return res
        if "API_KEY_INVALID" in errors[-1] or "HTTP 401" in errors[-1] or "HTTP 403" in errors[-1]:
            raise RuntimeError("Gemini key problem -> " + errors[-1])

    # Fallback: let Google tell us which models this key can use.
    try:
        avail = [m for m in available_models(key) if m not in tried and not any(s in m for s in SKIP)]
        flash = [m for m in avail if "flash" in m]
        gemma = [m for m in avail if "gemma" in m]
        for m in (flash + gemma)[:4]:
            res = attempt(m)
            if res:
                return res
    except Exception as e:
        errors.append(f"list models: {e}")
    raise RuntimeError("All Gemini models failed -> " + " | ".join(errors))


def ask(prompt):
    return ask_gemini(prompt)


## 3. Your subjects (edit this!)

In [25]:
from datetime import date

HOURS_PER_DAY = 4  # free study hours per day

subjects = [
    Subject("Data Mining",   date(2026, 11, 20), difficulty=3),  # 1 easy, 2 medium, 3 hard
    Subject("Cybersecurity", date(2026, 11, 15), difficulty=2),
    Subject("Computer Graphics", date(2026, 11, 25), difficulty=1),
]


## 4. Generate the schedule

In [26]:
import pandas as pd

rows = generate_schedule(subjects, HOURS_PER_DAY)
df = pd.DataFrame(rows)
display(df.pivot_table(index="date", columns="subject", values="hours", fill_value=0))
print("Total hours per subject:", summarize(rows))


subject,Computer Graphics,Cybersecurity,Data Mining
date,,,
2026-10-22,0.5,1.5,2.0
2026-10-23,0.5,1.5,2.0
2026-10-24,0.5,1.5,2.0
2026-10-25,0.5,1.5,2.0
2026-10-26,0.5,1.5,2.0
2026-10-27,0.5,1.5,2.0
2026-10-28,0.5,1.5,2.0
2026-10-29,0.5,1.5,2.0
2026-10-30,0.5,1.5,2.0


Total hours per subject: {'Data Mining': 62.5, 'Cybersecurity': 44.0, 'Computer Graphics': 33.5}


## 5. AI study tips (Gemini, with Gemma fallback)

In [27]:
for s in subjects:
    days = (s.exam_date - date.today()).days
    level = {1: "easy", 2: "medium", 3: "hard"}[s.difficulty]
    prompt = (f"Give 3 short, practical study tips for a university student preparing for "
              f"'{s.name}' (difficulty: {level}, exam in {days} days). Use bullet points.")
    try:
        model, tips = ask(prompt)
        print(f"\n### {s.name}  [{model}]\n{tips}")
    except Exception as e:
        print(f"\n{s.name}: AI skipped ({e})")



### Data Mining  [gemini-flash-lite-latest]
* **Master the math behind the algorithms:** Don't just memorize how Apriori, K-Means, or PageRank work conceptually. You must be able to manually calculate distance metrics, support/confidence, and entropy on paper under time pressure.
* **Code the core algorithms from scratch:** Implement decision trees, k-NN, or gradient descent in Python (NumPy/Pandas) without using high-level libraries like Scikit-Learn. Understanding the data structures and bottlenecks is heavily tested in hard exams.
* **Reverse-engineer past exams weekly:** Data mining exams love tricky edge cases (e.g., handling sparse matrices, overfitting in high dimensions, or dealing with noisy data). Find past papers immediately and practice applying theoretical concepts to messy, real-world scenarios.

### Cybersecurity  [gemini-flash-lite-latest]
Here are 3 practical study tips for your Cybersecurity exam:

*   **Prioritize High-Yield Frameworks:** Focus heavily on core conce

## 6. Download your plan

In [28]:
export_csv(rows, "study_plan.csv")
export_ics(rows, "study_plan.ics")  # import into Google Calendar
try:
    from google.colab import files
    files.download("study_plan.csv"); files.download("study_plan.ics")
except ImportError:
    print("Saved study_plan.csv and study_plan.ics")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>